In [13]:
import re
import pandas as pd

CONCORDANCE_CSV = "JJ113_image_data.csv"  
SEP = ";"
LABELSTUDIO_JSON_IN = "..\\List-of-zones\\Himanis_Seg_Actes_1200pxmin_JJ100-JJ118_labelstudio.json"   # à adapter
LABELSTUDIO_JSON_OUT = "..\\List-of-zones\\Himanis_Seg_Actes_1200pxmin_JJ100-JJ118_labelstudio_updated.json"  # à adapter


FILENAME_RE = re.compile(r'(JJ\d+)_0*(\d+)\.[A-Za-z]+$')

def parse_registre_ordre(filename: str):
    m = FILENAME_RE.search(filename) if isinstance(filename, str) else None
    if not m:
        return None, None
    return m.group(1), int(m.group(2))  # '0001' -> 1

df_conc = pd.read_csv(CONCORDANCE_CSV, sep=SEP, dtype=str)

concordance = {}
for _, row in df_conc.iterrows():
    registre, ordre = parse_registre_ordre(row["imageFileName"])
    if registre is not None:
        concordance[(registre, ordre)] = {"urlResizedImage": row["urlResizedImage"], "urlImage": row["urlImage"]}

print(f"{len(concordance)} entrée(s) de concordance chargée(s).")
print(concordance)


366 entrée(s) de concordance chargée(s).
{('JJ113', 1): {'urlResizedImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vmbfdedgtljq/full/1200,/0/default.jpg', 'urlImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vmbfdedgtljq/full/full/0/default.jpg'}, ('JJ113', 2): {'urlResizedImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vohk5ki93wko/full/1200,/0/default.jpg', 'urlImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vohk5ki93wko/full/full/0/default.jpg'}, ('JJ113', 3): {'urlResizedImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxd0dt9iha4s/full/1200,/0/default.jpg', 'urlImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vxd0dt9iha4s/full/full/0/default.jpg'}, ('JJ113', 4): {'urlResizedImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqh73y0pirdz/full/1200,/0/default.jpg', 'urlImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vqh73y0pirdz/full/full/0/default.jpg'}, ('JJ113', 5): {'urlResizedImage': 'https://iiif.irht.cnrs.fr/iiif/ark:/63955/vcijurnqbbop/full/1200,/0/default

In [ ]:
import json


with open(LABELSTUDIO_JSON_IN, encoding="utf-8") as f:
    tasks = json.load(f)

n_updated = 0
n_missing = 0
missing_keys = []

for task in tasks:
    data = task.get("data", {})
    registre = data.get("registre")
    ordre = data.get("ordre")
    if registre is None or ordre is None:
        continue
    try:
        ordre = int(ordre)
    except (TypeError, ValueError):
        continue

    key = (registre, ordre)
    entry = concordance.get(key)

    if entry is None:
        n_missing += 1
        missing_keys.append(key)
        continue

    if entry.get("urlResizedImage"):
        data["image"] = entry["urlResizedImage"]
        data["image_path"] = entry["urlImage"]
        n_updated += 1

with open(LABELSTUDIO_JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(tasks, f, ensure_ascii=False, indent=2)

print(f"✅ {n_updated} task(s) mise(s) à jour.")
print(f"⚠️ {n_missing} task(s) sans correspondance dans la concordance.")
if missing_keys:
    print("Exemples de clés manquantes :", missing_keys[:10])

✅ 365 task(s) mise(s) à jour.
⚠️ 7077 task(s) sans correspondance dans la concordance.
Exemples de clés manquantes : [('JJ100', 1), ('JJ100', 2), ('JJ100', 3), ('JJ100', 4), ('JJ100', 5), ('JJ100', 6), ('JJ100', 7), ('JJ100', 8), ('JJ100', 9), ('JJ100', 10)]
